# Radar Financeiro — processamento massivo em SQL

## 1. Preparação

Execute as células em ordem no ambiente corporativo. Os cálculos executam no Spark remoto; `%%time` mede a célula com materialização das etapas principais.


In [ ]:
from traceback import format_exc
try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal
    gerenciador_local = GerenciadorLocal(
        nome_sessao='radar-todos-sql',
        exibir_configuracao=False,
        ativar_logs=True,
    )
    spark = gerenciador_local.criar_sessao_spark(db2=True)
except Exception:
    print(format_exc())
    raise


In [ ]:
# Carregar os componentes corporativos no ambiente correspondente.
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


In [ ]:
%%time
%%spark
import os
from datetime import date

periodo = 1
FETCHSIZE = 10000
DATA_EXECUCAO = date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10]).isoformat()
conector_db2 = criar_conector_db2_spark(env=dict(os.environ))
resultado_validado = False
assert type(periodo) is int and 1 <= periodo <= 6, 'periodo deve ser inteiro entre 1 e 6.'
for objeto in spark.catalog.listTables():
    if objeto.isTemporary and objeto.name.startswith('rts_'):
        spark.sql(f"UNCACHE TABLE IF EXISTS {objeto.name}")
        spark.sql(f"DROP VIEW IF EXISTS {objeto.name}")

# Definir as referências da execução.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW rts_parametros AS
SELECT DATE('{DATA_EXECUCAO}') AS DT_EXEA,
       TRUNC(DATE('{DATA_EXECUCAO}'), 'MM') AS DT_MES_EXEA,
       ADD_MONTHS(DATE('{DATA_EXECUCAO}'), -1) AS DT_PUBLICO_INI,
       {periodo} AS PERIODO
""")
parametros = spark.sql("SELECT DT_PUBLICO_INI FROM rts_parametros").first()


## 2. Público e contexto


In [ ]:
%%time
%%spark
# Q1 — Buscar os registros que formam o público.
conector_db2.sql(f"""
SELECT CD_CLI, TS_INCL_TRAN, NR_CPF_CNPJ_TITR,
       NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND CD_TIP_PSS = 1
  AND TIMESTAMP(TS_INCL_TRAN) >= TIMESTAMP('{parametros["DT_PUBLICO_INI"]} 00:00:00')
  AND TIMESTAMP(TS_INCL_TRAN) < TIMESTAMP('{DATA_EXECUCAO} 00:00:00')
""", fetchsize=FETCHSIZE, query_timeout=900).createOrReplaceTempView('rts_q1')
spark.sql("CACHE TABLE rts_q1")
spark.sql("SELECT COUNT(*) AS QT_REGISTROS_Q1 FROM rts_q1").show()


In [ ]:
%%time
%%spark
# Formar CPF, conta única e referência do público.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_publico AS
WITH contas AS (
    SELECT *,
           NR_MCA_PCT_OPB = 999999999 AND CD_PRD = 6
           AND NR_AG_TITR IS NOT NULL AND CD_CT_TITR IS NOT NULL
           AND TRIM(CAST(CD_CT_TITR AS STRING)) <> '' AS CONTA_ELEGIVEL
    FROM rts_q1
), agrupado AS (
    SELECT CD_CLI,
           MAX(TS_INCL_TRAN) AS TS_INCL_TRAN_REF,
           COUNT(DISTINCT NR_CPF_CNPJ_TITR) AS QT_CPFS,
           CAST(MAX(NR_CPF_CNPJ_TITR) AS DECIMAL(14,0)) AS CPF,
           COUNT(DISTINCT CASE WHEN CONTA_ELEGIVEL
                 THEN NAMED_STRUCT('AG', NR_AG_TITR, 'CT', CD_CT_TITR) END) AS QT_CONTAS,
           TRIM(CAST(MAX(CASE WHEN CONTA_ELEGIVEL THEN NR_AG_TITR END) AS STRING)) AS AGENCIA,
           TRIM(CAST(MAX(CASE WHEN CONTA_ELEGIVEL THEN CD_CT_TITR END) AS STRING)) AS CONTA
    FROM contas
    GROUP BY CD_CLI
), normalizado AS (
    SELECT *,
           REGEXP_REPLACE(CONTA, '^0+', '') AS CONTA_SIGNIFICATIVA,
           QT_CONTAS = 1
           AND AGENCIA RLIKE '^[0-9]+$' AND CONTA RLIKE '^[0-9]+$'
           AND TRY_CAST(AGENCIA AS DECIMAL(38,0)) BETWEEN -2147483648 AND 2147483647
           AND LENGTH(REGEXP_REPLACE(CONTA, '^0+', '')) <= 11 AS CONTA_VALIDA
    FROM agrupado
)
SELECT CAST(CD_CLI AS INT) AS CD_CLI,
       CAST(TS_INCL_TRAN_REF AS TIMESTAMP) AS TS_INCL_TRAN_REF,
       CASE WHEN QT_CPFS = 1 THEN 'S' ELSE 'N' END AS FL_CPF_UNICO,
       CASE WHEN QT_CPFS = 1 THEN CPF END AS CD_CPF,
       CASE WHEN QT_CONTAS = 1 THEN 'S' ELSE 'N' END AS FL_CONTA_ELEGIVEL_UNICA,
       CASE WHEN CONTA_VALIDA THEN CAST(AGENCIA AS INT) END AS CD_UOR_CC_NORM,
       CASE WHEN CONTA_VALIDA
            THEN CAST(CASE WHEN CONTA_SIGNIFICATIVA = '' THEN '0'
                           ELSE CONTA_SIGNIFICATIVA END AS DECIMAL(11,0)) END AS NR_CC_NORM
FROM normalizado
""")
spark.sql("CACHE TABLE rts_publico")
spark.sql("""
SELECT ASSERT_TRUE(COUNT(*) > 0, 'Público vazio.'),
       ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT CD_CLI), 'Cliente nulo ou duplicado no público.')
FROM rts_publico
""").show()
spark.sql("UNCACHE TABLE rts_q1")


In [ ]:
%%time
%%spark
# Q2 — Buscar os registros de ciclo financeiro.
conector_db2.sql("""
SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC, TS_ULT_EXEA_PSQ
FROM DB2GFP.CT_GRDR_FNCO
""", fetchsize=FETCHSIZE, query_timeout=900).createOrReplaceTempView('rts_q2')
spark.sql("CACHE TABLE rts_q2")
spark.sql("SELECT COUNT(*) AS QT_REGISTROS_Q2 FROM rts_q2").show()


In [ ]:
%%time
%%spark
# Selecionar o ciclo da conta e aplicar o fallback.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_contexto_ciclo AS
WITH contas AS (
    SELECT DISTINCT CD_UOR_CC_NORM, NR_CC_NORM
    FROM rts_publico
    WHERE CD_UOR_CC_NORM IS NOT NULL AND NR_CC_NORM IS NOT NULL
), ordenado AS (
    SELECT c.*,
           ROW_NUMBER() OVER (
               PARTITION BY c.CD_UOR_CC, c.NR_CC
               ORDER BY c.TS_ULT_EXEA_PSQ DESC NULLS FIRST
           ) AS RN
    FROM rts_q2 c
    JOIN contas p ON c.CD_UOR_CC = p.CD_UOR_CC_NORM AND c.NR_CC = p.NR_CC_NORM
)
SELECT p.*,
       CAST(c.TS_ULT_EXEA_PSQ AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
       CAST(c.DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
       CAST(CASE WHEN p.CD_UOR_CC_NORM IS NULL OR p.NR_CC_NORM IS NULL THEN NULL
                 WHEN c.DD_INC_MM_CLC_BLC IS NULL THEN 1
                 ELSE c.DD_INC_MM_CLC_BLC END AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK
FROM rts_publico p
LEFT JOIN ordenado c
  ON c.CD_UOR_CC = p.CD_UOR_CC_NORM AND c.NR_CC = p.NR_CC_NORM AND c.RN = 1
""")
spark.sql("CACHE TABLE rts_contexto_ciclo")
spark.sql("""
SELECT ASSERT_TRUE(COUNT(CASE WHEN DD_INC_MM_CLC_BLC_FALLBACK NOT BETWEEN 1 AND 31
                             THEN 1 END) = 0, 'Dia de ciclo fora de 1..31.')
FROM rts_contexto_ciclo
""").show()
spark.sql("UNCACHE TABLE rts_q2")


In [ ]:
%%time
%%spark
# Q3 — Buscar os registros de renda na réplica Hive.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_q3 AS
SELECT NR_CPF_BASE_SRF, DT_INCL_REN_AVLD, VL_REN
FROM DB2DFE.REN_AVLD_PF
""")
spark.sql("CACHE TABLE rts_q3")
spark.sql("SELECT COUNT(*) AS QT_REGISTROS_Q3 FROM rts_q3").show()


In [ ]:
%%time
%%spark
# Selecionar a renda mais recente do CPF único.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_contexto_renda AS
WITH cpfs AS (
    SELECT DISTINCT CD_CPF FROM rts_publico WHERE CD_CPF IS NOT NULL
), ordenado AS (
    SELECT r.*,
           ROW_NUMBER() OVER (
               PARTITION BY r.NR_CPF_BASE_SRF
               ORDER BY r.DT_INCL_REN_AVLD DESC NULLS LAST
           ) AS RN
    FROM rts_q3 r
    JOIN cpfs p ON r.NR_CPF_BASE_SRF = p.CD_CPF
)
SELECT p.*,
       CAST(r.DT_INCL_REN_AVLD AS DATE) AS DT_REN_PRES_REF,
       CAST(r.VL_REN * x.PERIODO AS DECIMAL(17,2)) AS VL_REN_PRES
FROM rts_contexto_ciclo p
CROSS JOIN rts_parametros x
LEFT JOIN ordenado r ON p.CD_CPF = r.NR_CPF_BASE_SRF AND r.RN = 1
""")
spark.sql("CACHE TABLE rts_contexto_renda")
spark.sql("UNCACHE TABLE rts_q3")
spark.sql("UNCACHE TABLE rts_contexto_ciclo")


In [ ]:
%%time
%%spark
# Q4 — Buscar os perfis válidos até a data de execução.
conector_db2.sql(f"""
SELECT CD_CLI, DT_REF, CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI,
       CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
FROM DB2D1D.DVS_GRDR_FNCO_PF
WHERE DT_REF <= DATE('{DATA_EXECUCAO}')
""", fetchsize=FETCHSIZE, query_timeout=900).createOrReplaceTempView('rts_q4')
spark.sql("CACHE TABLE rts_q4")
spark.sql("SELECT COUNT(*) AS QT_REGISTROS_Q4 FROM rts_q4").show()


In [ ]:
%%time
%%spark
# Selecionar o perfil na maior referência e rejeitar empate.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_perfil AS
WITH referencias AS (
    SELECT f.*, MAX(f.DT_REF) OVER (PARTITION BY f.CD_CLI) AS MAIOR_DT
    FROM rts_q4 f
    JOIN rts_publico p ON f.CD_CLI = p.CD_CLI
)
SELECT CAST(CD_CLI AS INT) AS CD_CLI, CAST(DT_REF AS DATE) AS DT_REF_PRFL,
       CAST(CD_MAC_PRFL_CLI AS INT) AS CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI,
       CAST(CD_MIC_PRFL_CLI AS INT) AS CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI
FROM referencias
WHERE DT_REF = MAIOR_DT
""")
spark.sql("CACHE TABLE rts_perfil")
spark.sql("""
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT CD_CLI),
                   'Perfil empatado na maior DT_REF elegível.')
FROM rts_perfil
""").show()
spark.sql("UNCACHE TABLE rts_q4")


In [ ]:
%%time
%%spark
# Calcular a janela financeira e concluir o contexto de cada cliente.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_contexto AS
WITH referencia AS (
    SELECT p.*, x.DT_EXEA, x.DT_MES_EXEA, x.PERIODO,
           f.DT_REF_PRFL, f.CD_MAC_PRFL_CLI, f.NM_MAC_PRFL_CLI,
           f.CD_MIC_PRFL_CLI, f.NM_MIC_PRFL_CLI,
           TRUNC(TO_DATE(p.TS_INCL_TRAN_REF), 'MM') AS MES_REF
    FROM rts_contexto_renda p
    CROSS JOIN rts_parametros x
    LEFT JOIN rts_perfil f ON p.CD_CLI = f.CD_CLI
), aberto AS (
    SELECT *,
           CASE WHEN DD_INC_MM_CLC_BLC_FALLBACK IS NULL THEN CAST(NULL AS DATE)
                WHEN TS_INCL_TRAN_REF >= CAST(DATE_ADD(MES_REF,
                     LEAST(INT(DD_INC_MM_CLC_BLC_FALLBACK), DAY(LAST_DAY(MES_REF))) - 1) AS TIMESTAMP)
                THEN DATE_ADD(MES_REF,
                     LEAST(INT(DD_INC_MM_CLC_BLC_FALLBACK), DAY(LAST_DAY(MES_REF))) - 1)
                ELSE DATE_ADD(ADD_MONTHS(MES_REF, -1),
                     LEAST(INT(DD_INC_MM_CLC_BLC_FALLBACK), DAY(LAST_DAY(ADD_MONTHS(MES_REF, -1)))) - 1)
           END AS INICIO_ABERTO
    FROM referencia
)
SELECT CD_CLI, DT_EXEA, DT_MES_EXEA, TS_INCL_TRAN_REF,
       FL_CPF_UNICO, CD_CPF, FL_CONTA_ELEGIVEL_UNICA,
       TS_DD_INC_MM_CLC_BLC_REF, DD_INC_MM_CLC_BLC, DD_INC_MM_CLC_BLC_FALLBACK,
       DT_REN_PRES_REF, VL_REN_PRES, DT_REF_PRFL,
       CD_MAC_PRFL_CLI, NM_MAC_PRFL_CLI, CD_MIC_PRFL_CLI, NM_MIC_PRFL_CLI,
       CASE WHEN INICIO_ABERTO IS NOT NULL THEN
           DATE_ADD(ADD_MONTHS(TRUNC(INICIO_ABERTO, 'MM'), -PERIODO),
               LEAST(INT(DD_INC_MM_CLC_BLC_FALLBACK),
                     DAY(LAST_DAY(ADD_MONTHS(TRUNC(INICIO_ABERTO, 'MM'), -PERIODO)))) - 1)
       END AS DT_REF_INI,
       DATE_SUB(INICIO_ABERTO, 1) AS DT_REF_FIM
FROM aberto
""")
spark.sql("CACHE TABLE rts_contexto")
spark.sql("""
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT CD_CLI), 'Contexto duplicado.'),
       ASSERT_TRUE(COUNT(*) = (SELECT COUNT(*) FROM rts_publico), 'Contexto perdeu clientes.')
FROM rts_contexto
""").show()
spark.sql("UNCACHE TABLE rts_contexto_renda")
spark.sql("UNCACHE TABLE rts_perfil")


## 3. Movimentações


In [ ]:
%%time
%%spark
# Calcular o envelope temporal necessário à leitura.
limites = spark.sql("""
SELECT DATE_SUB(MIN(DT_REF_INI), 5) AS DT_MIN,
       DATE_ADD(MAX(DT_REF_FIM), 5) AS DT_MAX
FROM rts_contexto
""").first()


In [ ]:
%%time
%%spark
# Q5 — Buscar movimentos para as janelas e a reconciliação.
conector_db2.sql(f"""
SELECT NR_TRAN_INST_PCT, CD_CLI, DT_TRAN, CD_NTZ_CTB_TRAN,
       CD_CTGR_TRAN_OGNL, CD_TIP_MOE_CRR, VL_TRAN, NR_MCA_PCT_OPB
FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI IS NOT NULL
  AND CD_EST_TRAN_INST = 0
  AND DT_TRAN >= {("DATE('" + str(limites["DT_MIN"]) + "')") if limites["DT_MIN"] is not None else "CAST(NULL AS DATE)"}
  AND DT_TRAN <= {("DATE('" + str(limites["DT_MAX"]) + "')") if limites["DT_MAX"] is not None else "CAST(NULL AS DATE)"}
  AND (CD_NTZ_CTB_TRAN = 'C' OR (CD_NTZ_CTB_TRAN = 'D' AND IN_VSLO_CSM = 'S'))
""", fetchsize=FETCHSIZE, query_timeout=900).createOrReplaceTempView('rts_q5')
spark.sql("CACHE TABLE rts_q5")
spark.sql("SELECT COUNT(*) AS QT_REGISTROS_Q5 FROM rts_q5").show()


In [ ]:
%%time
%%spark
# Aplicar a janela individual ampliada e identificar os movimentos oficiais.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_movimentos AS
SELECT CAST(m.NR_TRAN_INST_PCT AS BIGINT) AS NR_TRAN_INST_PCT,
       p.CD_CLI, CAST(m.DT_TRAN AS DATE) AS DT_TRAN,
       m.CD_NTZ_CTB_TRAN, m.CD_CTGR_TRAN_OGNL, m.CD_TIP_MOE_CRR,
       CAST(m.VL_TRAN AS DECIMAL(15,2)) AS VL_TRAN, m.NR_MCA_PCT_OPB,
       CASE WHEN m.DT_TRAN BETWEEN p.DT_REF_INI AND p.DT_REF_FIM
            THEN 'S' ELSE 'N' END AS IN_JANELA
FROM rts_q5 m
JOIN rts_contexto p ON m.CD_CLI = p.CD_CLI
WHERE m.DT_TRAN BETWEEN DATE_SUB(p.DT_REF_INI, 5) AND DATE_ADD(p.DT_REF_FIM, 5)
""")
spark.sql("CACHE TABLE rts_movimentos")
spark.sql("""
SELECT ASSERT_TRUE(COUNT(*) = COUNT(NR_TRAN_INST_PCT), 'Movimento sem identidade.'),
       ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT NAMED_STRUCT('CLI', CD_CLI, 'ID', NR_TRAN_INST_PCT)),
                   'Movimento duplicado por cliente.')
FROM rts_movimentos
""").show()
spark.sql("UNCACHE TABLE rts_q5")


## 4. Reconciliação


In [ ]:
%%time
%%spark
# Reconciliar pares exatos pela mesma busca de caminhos aumentantes do Individual.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_pares_exatos AS
WITH creditos AS (
    SELECT CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA,
           SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(
               'id', NR_TRAN_INST_PCT, 'banco', NR_MCA_PCT_OPB))) AS C
    FROM rts_movimentos
    WHERE CD_NTZ_CTB_TRAN = 'C' AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL AND CD_TIP_MOE_CRR IS NOT NULL
    GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
), debitos AS (
    SELECT CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA,
           SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(
               'id', NR_TRAN_INST_PCT, 'banco', NR_MCA_PCT_OPB))) AS D
    FROM rts_movimentos
    WHERE CD_NTZ_CTB_TRAN = 'D' AND DT_TRAN IS NOT NULL
      AND VL_TRAN IS NOT NULL AND CD_TIP_MOE_CRR IS NOT NULL
    GROUP BY CD_CLI, DT_TRAN, VL_TRAN, CD_TIP_MOE_CRR, IN_JANELA
), grupos AS (
    SELECT c.CD_CLI, c.IN_JANELA, c.C, d.D
    FROM creditos c
    JOIN debitos d ON c.CD_CLI = d.CD_CLI AND c.DT_TRAN = d.DT_TRAN
                  AND c.VL_TRAN = d.VL_TRAN AND c.CD_TIP_MOE_CRR = d.CD_TIP_MOE_CRR
                  AND c.IN_JANELA = d.IN_JANELA
), pareados AS (
    SELECT CD_CLI, IN_JANELA, C, D,
           AGGREGATE(
             SEQUENCE(1, SIZE(C)),
             NAMED_STRUCT('c_d', ARRAY_REPEAT(0, SIZE(C)), 'd_c', ARRAY_REPEAT(0, SIZE(D))),
             (pares, credito) ->
               AGGREGATE(
                 SEQUENCE(1, SIZE(C)),
                 NAMED_STRUCT(
                   'fila', ARRAY(credito),
                   'visitados', CAST(ARRAY() AS ARRAY<INT>),
                   'anterior', ARRAY_REPEAT(0, SIZE(D)),
                   'livre', 0, 'pos', 1),
                 (busca, passo) ->
                   CASE WHEN busca.livre <> 0 OR busca.pos > SIZE(busca.fila) THEN busca
                   ELSE AGGREGATE(
                     SEQUENCE(1, SIZE(D)),
                     busca,
                     (b, debito) ->
                       CASE WHEN b.livre = 0
                                  AND NOT ARRAY_CONTAINS(b.visitados, debito)
                                  AND ELEMENT_AT(C, ELEMENT_AT(b.fila, b.pos)).banco IS NOT NULL
                                  AND ELEMENT_AT(D, debito).banco IS NOT NULL
                                  AND ELEMENT_AT(C, ELEMENT_AT(b.fila, b.pos)).banco
                                      <> ELEMENT_AT(D, debito).banco
                       THEN NAMED_STRUCT(
                         'fila', CASE WHEN ELEMENT_AT(pares.d_c, debito) <> 0
                                           AND NOT ARRAY_CONTAINS(b.fila, ELEMENT_AT(pares.d_c, debito))
                                      THEN CONCAT(b.fila, ARRAY(ELEMENT_AT(pares.d_c, debito)))
                                      ELSE b.fila END,
                         'visitados', CONCAT(b.visitados, ARRAY(debito)),
                         'anterior', TRANSFORM(b.anterior,
                             (a, indice) -> CASE WHEN indice + 1 = debito
                                                THEN ELEMENT_AT(b.fila, b.pos) ELSE a END),
                         'livre', CASE WHEN ELEMENT_AT(pares.d_c, debito) = 0 THEN debito ELSE 0 END,
                         'pos', b.pos)
                       ELSE b END,
                     avancada -> NAMED_STRUCT(
                         'fila', avancada.fila, 'visitados', avancada.visitados,
                         'anterior', avancada.anterior, 'livre', avancada.livre,
                         'pos', avancada.pos + 1)
                   ) END,
                 achado ->
                   AGGREGATE(
                     SEQUENCE(1, SIZE(C)),
                     NAMED_STRUCT('c_d', pares.c_d, 'd_c', pares.d_c, 'debito', achado.livre),
                     (troca, passo_retorno) ->
                       CASE WHEN troca.debito = 0 THEN troca
                       ELSE NAMED_STRUCT(
                         'c_d', TRANSFORM(troca.c_d,
                             (d_atual, indice) ->
                               CASE WHEN indice + 1 = ELEMENT_AT(achado.anterior, troca.debito)
                                    THEN troca.debito ELSE d_atual END),
                         'd_c', TRANSFORM(troca.d_c,
                             (c_atual, indice) ->
                               CASE WHEN indice + 1 = troca.debito
                                    THEN ELEMENT_AT(achado.anterior, troca.debito) ELSE c_atual END),
                         'debito', ELEMENT_AT(troca.c_d, ELEMENT_AT(achado.anterior, troca.debito))
                       ) END,
                     pronto -> NAMED_STRUCT('c_d', pronto.c_d, 'd_c', pronto.d_c)
                   )
               )
           ) AS PARES
    FROM grupos
)
SELECT CD_CLI, IN_JANELA,
       ELEMENT_AT(C, credito).id AS ID_CREDITO,
       ELEMENT_AT(D, ELEMENT_AT(PARES.c_d, credito)).id AS ID_DEBITO
FROM pareados
LATERAL VIEW EXPLODE(FILTER(SEQUENCE(1, SIZE(C)),
    indice -> ELEMENT_AT(PARES.c_d, indice) <> 0)) ids AS credito
""")
spark.sql("CACHE TABLE rts_pares_exatos")
spark.sql("SELECT COUNT(*) AS QT_PARES_EXATOS FROM rts_pares_exatos").show()


In [ ]:
%%time
%%spark
# Remover os movimentos consumidos pelos pares exatos.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_ids_exatos AS
SELECT CD_CLI, ID_CREDITO AS NR_TRAN_INST_PCT FROM rts_pares_exatos
UNION ALL
SELECT CD_CLI, ID_DEBITO AS NR_TRAN_INST_PCT FROM rts_pares_exatos
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_residual AS
SELECT m.*
FROM rts_movimentos m
LEFT ANTI JOIN rts_ids_exatos e
  ON m.CD_CLI = e.CD_CLI AND m.NR_TRAN_INST_PCT = e.NR_TRAN_INST_PCT
""")
spark.sql("CACHE TABLE rts_residual")
spark.sql("UNCACHE TABLE rts_movimentos")


In [ ]:
%%time
%%spark
# Reconciliar bordas com a programação dinâmica e os desempates do Individual.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_pares_borda AS
WITH dentro AS (
    SELECT CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN AS NTZ_DENTRO,
           SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(
               'dt', DT_TRAN, 'id', NR_TRAN_INST_PCT, 'banco', NR_MCA_PCT_OPB))) AS DENTRO
    FROM rts_residual
    WHERE IN_JANELA = 'S' AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR, CD_NTZ_CTB_TRAN
), fora AS (
    SELECT CD_CLI, VL_TRAN, CD_TIP_MOE_CRR,
           CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END AS NTZ_DENTRO,
           SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(
               'dt', DT_TRAN, 'id', NR_TRAN_INST_PCT, 'banco', NR_MCA_PCT_OPB))) AS FORA
    FROM rts_residual
    WHERE IN_JANELA = 'N' AND CD_NTZ_CTB_TRAN IN ('C', 'D')
    GROUP BY CD_CLI, VL_TRAN, CD_TIP_MOE_CRR,
             CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 'D' ELSE 'C' END
), grupos AS (
    SELECT d.CD_CLI, d.DENTRO, f.FORA,
           NAMED_STRUCT(
             'neg_qt', 0, 'custo', CAST(0 AS BIGINT),
             'ids', CAST(ARRAY() AS ARRAY<STRUCT<dentro:BIGINT,fora:BIGINT>>),
             'pares', CAST(ARRAY() AS ARRAY<STRUCT<
                 NR_TRAN_DENTRO:BIGINT,NR_TRAN_FORA:BIGINT,
                 DT_TRAN_DENTRO:DATE,DT_TRAN_FORA:DATE,DIF_DIAS:INT>>)
           ) AS VAZIO
    FROM dentro d
    JOIN fora f ON d.CD_CLI = f.CD_CLI AND d.VL_TRAN = f.VL_TRAN
               AND d.CD_TIP_MOE_CRR = f.CD_TIP_MOE_CRR AND d.NTZ_DENTRO = f.NTZ_DENTRO
), pareados AS (
    SELECT CD_CLI,
           AGGREGATE(
             SEQUENCE(SIZE(DENTRO), 1, -1),
             ARRAY_REPEAT(VAZIO, SIZE(FORA) + 1),
             (anterior, i) ->
               AGGREGATE(
                 SEQUENCE(SIZE(FORA), 1, -1),
                 ARRAY(VAZIO),
                 (atual, j) ->
                   CONCAT(
                     ARRAY(ARRAY_MIN(ARRAY(
                       ELEMENT_AT(anterior, j),
                       ELEMENT_AT(atual, 1),
                       CASE WHEN ABS(DATEDIFF(ELEMENT_AT(DENTRO, i).dt,
                                              ELEMENT_AT(FORA, j).dt)) BETWEEN 1 AND 5
                                  AND ELEMENT_AT(DENTRO, i).banco IS NOT NULL
                                  AND ELEMENT_AT(FORA, j).banco IS NOT NULL
                                  AND ELEMENT_AT(DENTRO, i).banco <> ELEMENT_AT(FORA, j).banco
                       THEN NAMED_STRUCT(
                         'neg_qt', ELEMENT_AT(anterior, j + 1).neg_qt - 1,
                         'custo', ELEMENT_AT(anterior, j + 1).custo
                                  + CAST(ABS(DATEDIFF(ELEMENT_AT(DENTRO, i).dt,
                                                     ELEMENT_AT(FORA, j).dt)) AS BIGINT),
                         'ids', CONCAT(ARRAY(NAMED_STRUCT(
                             'dentro', ELEMENT_AT(DENTRO, i).id,
                             'fora', ELEMENT_AT(FORA, j).id)), ELEMENT_AT(anterior, j + 1).ids),
                         'pares', CONCAT(ARRAY(NAMED_STRUCT(
                             'NR_TRAN_DENTRO', ELEMENT_AT(DENTRO, i).id,
                             'NR_TRAN_FORA', ELEMENT_AT(FORA, j).id,
                             'DT_TRAN_DENTRO', ELEMENT_AT(DENTRO, i).dt,
                             'DT_TRAN_FORA', ELEMENT_AT(FORA, j).dt,
                             'DIF_DIAS', ABS(DATEDIFF(ELEMENT_AT(DENTRO, i).dt,
                                                     ELEMENT_AT(FORA, j).dt))
                         )), ELEMENT_AT(anterior, j + 1).pares)
                       )
                       ELSE ELEMENT_AT(anterior, j) END
                     ))),
                     atual
                   )
               ),
             linha -> ELEMENT_AT(linha, 1).pares
           ) AS PARES
    FROM grupos
)
SELECT CD_CLI, par.NR_TRAN_DENTRO, par.NR_TRAN_FORA,
       par.DT_TRAN_DENTRO, par.DT_TRAN_FORA, par.DIF_DIAS
FROM pareados
LATERAL VIEW EXPLODE(PARES) pares_expandidos AS par
""")
spark.sql("CACHE TABLE rts_pares_borda")
spark.sql("SELECT COUNT(*) AS QT_PARES_BORDA FROM rts_pares_borda").show()


In [ ]:
%%time
%%spark
# Validar o consumo único e obter os movimentos efetivos.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_ids_borda AS
SELECT CD_CLI, NR_TRAN_DENTRO AS NR_TRAN_INST_PCT FROM rts_pares_borda
UNION ALL
SELECT CD_CLI, NR_TRAN_FORA AS NR_TRAN_INST_PCT FROM rts_pares_borda
""")
spark.sql("""
WITH consumidos AS (
    SELECT CD_CLI, NR_TRAN_INST_PCT FROM rts_ids_exatos
    UNION ALL
    SELECT CD_CLI, NR_TRAN_INST_PCT FROM rts_ids_borda
)
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT NAMED_STRUCT('CLI', CD_CLI, 'ID', NR_TRAN_INST_PCT)),
                   'Reconciliação reutilizou uma transação.')
FROM consumidos
""").show()
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_efetivos AS
SELECT m.*
FROM rts_residual m
LEFT ANTI JOIN rts_ids_borda b
  ON m.CD_CLI = b.CD_CLI AND m.NR_TRAN_INST_PCT = b.NR_TRAN_INST_PCT
WHERE m.IN_JANELA = 'S'
""")


In [ ]:
%%time
%%spark
# Classificar os movimentos efetivos pelo mapa oficial.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_categorias AS
SELECT * FROM VALUES
    (CAST(NULL AS STRING), 0, 'Sem categoria', 0, 'Sem categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 1, 'Salário', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 2, 'Vale Alimentação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 3, 'Restituição de IR', 0, 'Não pertence', 2, 'Estorno', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 4, 'Bonificação', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('C', 1, 'Receitas', 5, 'Outros Rendimentos', 0, 'Não pertence', 1, 'Renda', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 6, 'Água', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 7, 'Eletricidade e Gás', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 9, 'Compra de Imóvel', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 10, 'Aluguel e Condomínio', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 11, 'Móveis e Utensílios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 12, 'Serviços e Manutenção', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 13, 'Empregados', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 14, 'Animais e Pets', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 15, 'Educação Superior', 1, 'Pagamentos efetuados', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 16, 'Colégio', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 17, 'Idiomas', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 18, 'Publicações e Papelaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 3, 'Educação', 20, 'Outros Gastos, Educação', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 21, 'Viagens e Lazer', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 22, 'Esportes e Academia', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 25, 'Cultura e Entretenimento', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 27, 'Plano de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 28, 'Serviços de Saúde', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 29, 'Dentista', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 5, 'Saúde', 30, 'Farmácias e Drogarias', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 32, 'Feira e Supermercado', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 6, 'Alimentação', 35, 'Bar, Rest. e Padaria', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 36, 'Compra de Veículo', 2, 'Bens e direitos', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 37, 'Combustível', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 38, 'Estacionamento e Pedágio', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 39, 'Seguro de Veículo', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 40, 'Serviços e Manutenção', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 7, 'Transporte', 41, 'Transporte Urbano e Apps', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 42, 'Vestuário e Acessórios', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 43, 'Cuidado Pessoal e Beleza', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 44, 'Compras Diversas', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 45, 'Pensão Alimentícia', 1, 'Pagamentos efetuados', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 46, 'Seguros e Previdência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 47, 'Doação', 4, 'Doações efetuadas', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 48, 'Gasto com Familiares', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 49, 'Presentes', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 51, 'Telefonia e Internet', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 9, 'Comunicação', 53, 'Assinatura TV e Streaming', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 54, 'IPTU', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 55, 'IPVA e Gastos Detran', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 56, 'Imposto de Renda', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 57, 'ISS(Imposto sobre Serviços)', 0, 'Não pertence', 6, 'Essenciais', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 58, 'GPS(Guia de Previdência Social)', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 59, 'Serviços Financeiros', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 60, 'Serviços Diversos', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 4, 'Lazer', 61, 'Jogos e Loterias', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    (CAST(NULL AS STRING), 0, 'Sem categoria', 83, 'Sem Categoria', 0, 'Não pertence', 0, 'Outras Entradas', 'N', 'S', 'S'),
    ('D', 12, 'Fatura', 111, 'Cartão de Crédito', 0, 'Não pertence', 9, 'Obrigações', 'N', 'N', 'N'),
    ('D', 11, 'Outros', 279, 'Gastos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('C', 14, 'Agro', 300, 'Receitas Agro', 0, 'Não pertence', 1, 'Renda', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 310, 'Criações', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 330, 'Cultivos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 350, 'Insumos', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 14, 'Agro', 370, 'Apoio Produtivo', 0, 'Não pertence', 5, 'Indeterminado', 'S', 'N', 'N'),
    ('D', 10, 'Tarifas e impostos', 3787, 'IOF', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 10, 'Tarifas e impostos', 3788, 'Encargos e Tarifas', 0, 'Não pertence', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 2, 'Casa', 3790, 'Seguro Residencial', 0, 'Não pertence', 7, 'Não Essenciais', 'N', 'S', 'S'),
    ('D', 8, 'Despesas Pessoais', 4417, 'Empréstimos e Prestações', 3, 'Dívidas e ônus reais', 9, 'Obrigações', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39434, 'Cheque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39435, 'Saque', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39436, 'Transferência', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 11, 'Outros', 39437, 'Boletos Diversos', 0, 'Não pertence', 5, 'Indeterminado', 'N', 'S', 'S'),
    ('D', 13, 'Investimentos', 448977, 'Aplicação', 0, 'Não pertence', 8, 'Futuro', 'N', 'S', 'N'),
    ('C', 13, 'Investimentos', 448978, 'Resgate de Investimentos', 0, 'Não pertence', 3, 'Resgate', 'N', 'S', 'N')
AS c(TIPO, CD_GRUPO, TX_GRUPO, CD_CATEGORIA, TX_CATEGORIA,
     CD_IR, TX_IR, CD_CLASS_RADAR, TX_CLASS_RADAR,
     IN_AGRO, IN_PARTICIPA_CALCULO, IN_PARTICIPA_ORCAMENTO)
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_classificados AS
SELECT m.*,
       c.CD_CATEGORIA IS NOT NULL AS TEM_CATEGORIA,
       COALESCE(c.CD_CLASS_RADAR, 0) AS CD_CLASS_RADAR,
       COALESCE(c.IN_AGRO, 'N') AS IN_AGRO,
       COALESCE(c.IN_PARTICIPA_CALCULO, 'N') AS IN_PARTICIPA_CALCULO,
       COALESCE(c.IN_PARTICIPA_ORCAMENTO, 'N') AS IN_PARTICIPA_ORCAMENTO
FROM rts_efetivos m
LEFT JOIN rts_categorias c
  ON m.CD_CTGR_TRAN_OGNL = c.CD_CATEGORIA AND m.CD_NTZ_CTB_TRAN = c.TIPO
""")
spark.sql("CACHE TABLE rts_classificados")
spark.sql("UNCACHE TABLE rts_residual")
spark.sql("UNCACHE TABLE rts_pares_exatos")
spark.sql("UNCACHE TABLE rts_pares_borda")


## 5. Agregações e motor


In [ ]:
%%time
%%spark
# Agregar os valores financeiros preservando todos os clientes do público.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_financeiro AS
SELECT p.CD_CLI,
       CASE WHEN COUNT(m.NR_TRAN_INST_PCT) = 0 THEN NULL
            WHEN COUNT(DISTINCT m.CD_TIP_MOE_CRR) = 1 AND MAX(m.CD_TIP_MOE_CRR) = 'BRL'
            THEN 'S' ELSE 'N' END AS FL_SOMENTE_BRL,
       CASE WHEN COUNT(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' THEN 1 END) = 0 THEN NULL
            WHEN COUNT(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.IN_AGRO = 'S' THEN 1 END) > 0
            THEN 'S' ELSE 'N' END AS FL_TEM_MOV_AGRO,
       COUNT(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' THEN 1 END) AS QT_TRANS_TOTAL,
       SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C' THEN 1
                WHEN m.CD_TIP_MOE_CRR = 'BRL' THEN 0 END) AS QT_TRANS_ENT,
       SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D' THEN 1
                WHEN m.CD_TIP_MOE_CRR = 'BRL' THEN 0 END) AS QT_TRANS_SAI,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.CD_CLASS_RADAR = 1 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_REN,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.CD_CLASS_RADAR = 2 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_EST,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.CD_CLASS_RADAR = 3 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_RESG,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.CD_CLASS_RADAR = 0 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_OUT,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.CD_CLASS_RADAR = 4 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_CRED,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D'
                                  AND m.CD_CLASS_RADAR = 5 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_IND,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D'
                                  AND m.CD_CLASS_RADAR = 6 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_ESS,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D'
                                  AND m.CD_CLASS_RADAR = 7 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D'
                                  AND m.CD_CLASS_RADAR = 8 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_FUT,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D'
                                  AND m.CD_CLASS_RADAR = 9 AND m.IN_PARTICIPA_CALCULO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_OBR,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.IN_PARTICIPA_ORCAMENTO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_ENT_TOTAL,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'D'
                                  AND m.IN_PARTICIPA_ORCAMENTO = 'S'
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS VL_SAI_TOTAL,
       CAST(COALESCE(SUM(CASE WHEN m.CD_TIP_MOE_CRR = 'BRL' AND m.CD_NTZ_CTB_TRAN = 'C'
                                  AND m.TEM_CATEGORIA
                             THEN m.VL_TRAN END), 0.00) AS DECIMAL(25,2)) AS ENTRADAS_REALIZADAS
FROM rts_publico p
LEFT JOIN rts_classificados m ON p.CD_CLI = m.CD_CLI
GROUP BY p.CD_CLI
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_base AS
SELECT p.*, f.FL_SOMENTE_BRL, f.FL_TEM_MOV_AGRO,
       f.QT_TRANS_TOTAL, f.QT_TRANS_ENT, f.QT_TRANS_SAI,
       f.VL_ENT_TOTAL AS VL_TRANS_ENT, f.VL_SAI_TOTAL AS VL_TRANS_SAI,
       f.VL_ENT_REN, f.VL_ENT_EST, f.VL_ENT_RESG, f.VL_ENT_OUT, f.VL_ENT_CRED, f.VL_ENT_TOTAL,
       f.VL_SAI_IND, f.VL_SAI_ESS, f.VL_SAI_NAO_ESS, f.VL_SAI_FUT, f.VL_SAI_OBR, f.VL_SAI_TOTAL,
       f.ENTRADAS_REALIZADAS
FROM rts_contexto p
LEFT JOIN rts_financeiro f ON p.CD_CLI = f.CD_CLI
""")
spark.sql("CACHE TABLE rts_base")
spark.sql("SELECT COUNT(*) AS QT_CLIENTES_AGREGADOS FROM rts_base").show()
spark.sql("UNCACHE TABLE rts_classificados")
spark.sql("UNCACHE TABLE rts_contexto")


In [ ]:
%%time
%%spark
# Aplicar o motor e os cenários com arredondamento HALF_UP exato em centavos.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_motor AS
WITH bases AS (
    SELECT b.*, cenario, base_orc, base_pc
    FROM rts_base b
    LATERAL VIEW INLINE(ARRAY(
        NAMED_STRUCT('cenario', 'OFICIAL',
                     'base_orc', VL_ENT_TOTAL,
                     'base_pc', CAST(VL_REN_PRES AS DECIMAL(25,2))),
        NAMED_STRUCT('cenario', 'RENDA_PRESUMIDA',
                     'base_orc', CASE WHEN DT_REF_INI IS NOT NULL AND DT_REF_FIM IS NOT NULL
                                      THEN CAST(VL_REN_PRES AS DECIMAL(25,2)) END,
                     'base_pc', CASE WHEN DT_REF_INI IS NOT NULL AND DT_REF_FIM IS NOT NULL
                                     THEN CAST(VL_REN_PRES AS DECIMAL(25,2)) END),
        NAMED_STRUCT('cenario', 'ENTRADAS_REALIZADAS',
                     'base_orc', CASE WHEN DT_REF_INI IS NOT NULL AND DT_REF_FIM IS NOT NULL
                                      THEN ENTRADAS_REALIZADAS END,
                     'base_pc', CASE WHEN DT_REF_INI IS NOT NULL AND DT_REF_FIM IS NOT NULL
                                     THEN ENTRADAS_REALIZADAS END)
    )) cenarios AS cenario, base_orc, base_pc
), centavos AS (
    SELECT *,
           CAST(base_orc - VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_RES_ORC,
           ARRAY(
             NAMED_STRUCT(
               'n', CAST(ABS(VL_SAI_TOTAL) * 100000000 AS DECIMAL(31,0)),
               'd', CASE WHEN QT_TRANS_TOTAL <> 0 THEN CAST(ABS(base_orc) * 100 AS DECIMAL(25,0)) END,
               'sinal', CASE WHEN (VL_SAI_TOTAL < 0 AND base_orc > 0)
                                  OR (VL_SAI_TOTAL > 0 AND base_orc < 0) THEN -1 ELSE 1 END),
             NAMED_STRUCT(
               'n', CAST(ABS(VL_SAI_IND) * 100000000 AS DECIMAL(31,0)),
               'd', CASE WHEN base_pc > 0 THEN CAST(base_pc * 100 AS DECIMAL(25,0)) END,
               'sinal', CASE WHEN VL_SAI_IND < 0 THEN -1 ELSE 1 END),
             NAMED_STRUCT(
               'n', CAST(ABS(VL_SAI_ESS) * 100000000 AS DECIMAL(31,0)),
               'd', CASE WHEN base_pc > 0 THEN CAST(base_pc * 100 AS DECIMAL(25,0)) END,
               'sinal', CASE WHEN VL_SAI_ESS < 0 THEN -1 ELSE 1 END),
             NAMED_STRUCT(
               'n', CAST(ABS(VL_SAI_NAO_ESS) * 100000000 AS DECIMAL(31,0)),
               'd', CASE WHEN base_pc > 0 THEN CAST(base_pc * 100 AS DECIMAL(25,0)) END,
               'sinal', CASE WHEN VL_SAI_NAO_ESS < 0 THEN -1 ELSE 1 END),
             NAMED_STRUCT(
               'n', CAST(ABS(VL_SAI_FUT) * 100000000 AS DECIMAL(31,0)),
               'd', CASE WHEN base_pc > 0 THEN CAST(base_pc * 100 AS DECIMAL(25,0)) END,
               'sinal', CASE WHEN VL_SAI_FUT < 0 THEN -1 ELSE 1 END),
             NAMED_STRUCT(
               'n', CAST(ABS(VL_SAI_OBR) * 100000000 AS DECIMAL(31,0)),
               'd', CASE WHEN base_pc > 0 THEN CAST(base_pc * 100 AS DECIMAL(25,0)) END,
               'sinal', CASE WHEN VL_SAI_OBR < 0 THEN -1 ELSE 1 END)
           ) AS FRACOES
    FROM bases
), arredondado AS (
    SELECT *,
           TRANSFORM(FRACOES, f ->
             CASE WHEN f.n IS NULL OR f.d IS NULL OR f.d = 0
                        OR f.n >= f.d * CAST(1000000000 AS DECIMAL(10,0))
                  THEN CAST(NULL AS BIGINT)
                  ELSE f.sinal * (
                      CAST((f.n - PMOD(f.n, f.d)) / f.d AS BIGINT)
                      + CASE WHEN PMOD(f.n, f.d) * 2 >= f.d THEN 1 ELSE 0 END)
             END
           ) AS MICROS
    FROM centavos
), percentuais AS (
    SELECT *,
           CASE WHEN ABS(ELEMENT_AT(MICROS, 1)) < 1000000000
                THEN CAST(CAST(ELEMENT_AT(MICROS, 1) AS DECIMAL(15,0))
                          / CAST(1000000 AS DECIMAL(7,0)) AS DECIMAL(9,6)) END AS PC_SAI_ENT,
           CASE WHEN ABS(ELEMENT_AT(MICROS, 2)) < 1000000000
                THEN CAST(CAST(ELEMENT_AT(MICROS, 2) AS DECIMAL(15,0))
                          / CAST(1000000 AS DECIMAL(7,0)) AS DECIMAL(9,6)) END AS PC_SAI_IND,
           CASE WHEN ABS(ELEMENT_AT(MICROS, 3)) < 1000000000
                THEN CAST(CAST(ELEMENT_AT(MICROS, 3) AS DECIMAL(15,0))
                          / CAST(1000000 AS DECIMAL(7,0)) AS DECIMAL(9,6)) END AS PC_SAI_ESS,
           CASE WHEN ABS(ELEMENT_AT(MICROS, 4)) < 1000000000
                THEN CAST(CAST(ELEMENT_AT(MICROS, 4) AS DECIMAL(15,0))
                          / CAST(1000000 AS DECIMAL(7,0)) AS DECIMAL(9,6)) END AS PC_SAI_NAO_ESS,
           CASE WHEN ABS(ELEMENT_AT(MICROS, 5)) < 1000000000
                THEN CAST(CAST(ELEMENT_AT(MICROS, 5) AS DECIMAL(15,0))
                          / CAST(1000000 AS DECIMAL(7,0)) AS DECIMAL(9,6)) END AS PC_SAI_FUT,
           CASE WHEN ABS(ELEMENT_AT(MICROS, 6)) < 1000000000
                THEN CAST(CAST(ELEMENT_AT(MICROS, 6) AS DECIMAL(15,0))
                          / CAST(1000000 AS DECIMAL(7,0)) AS DECIMAL(9,6)) END AS PC_SAI_OBR
    FROM arredondado
), faixas AS (
    SELECT *,
           CASE WHEN PC_SAI_ENT IS NULL THEN NULL
                WHEN PC_SAI_ENT BETWEEN 0.950000 AND 1.050000 THEN 0
                WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
                WHEN PC_SAI_ENT > 1.250000 THEN 2
                WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
                ELSE 4 END AS CD_FAIXA_ORC
    FROM percentuais
), pontos AS (
    SELECT *,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR base_pc IS NULL THEN NULL
                WHEN base_pc <= 0 THEN 0
                ELSE CASE WHEN PC_SAI_IND > 0.750000 THEN 99 ELSE 0 END END AS NR_PONT_CONC_IND,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR base_pc IS NULL THEN NULL
                WHEN base_pc <= 0 THEN 0
                ELSE CASE WHEN PC_SAI_ESS < 0.500000 THEN 0 WHEN PC_SAI_ESS < 0.750000 THEN 1 ELSE 2 END END AS NR_PONT_CONC_ESS,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR base_pc IS NULL THEN NULL
                WHEN base_pc <= 0 THEN 0
                ELSE CASE WHEN PC_SAI_NAO_ESS < 0.300000 THEN 0 WHEN PC_SAI_NAO_ESS < 0.450000 THEN 1 ELSE 2 END END AS NR_PONT_CONC_NAO_ESS,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR base_pc IS NULL THEN NULL
                WHEN base_pc <= 0 THEN 0
                ELSE CASE WHEN PC_SAI_FUT >= 0.300000 THEN 0 WHEN PC_SAI_FUT >= 0.200000 THEN 1 ELSE 2 END END AS NR_PONT_CONC_FUT,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR base_pc IS NULL THEN NULL
                WHEN base_pc <= 0 THEN 0
                ELSE CASE WHEN PC_SAI_OBR < 0.300000 THEN 0 WHEN PC_SAI_OBR < 0.450000 THEN 1 ELSE 2 END END AS NR_PONT_CONC_OBR,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_ORC_IND,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
                WHEN CD_FAIXA_ORC = 2 THEN 2 WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
                ELSE 0 END AS NR_PONT_ORC_ESS,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
                WHEN CD_FAIXA_ORC = 2 THEN 2 WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
                ELSE 0 END AS NR_PONT_ORC_NAO_ESS,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
                WHEN CD_FAIXA_ORC = 4 THEN 2 WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
                ELSE 0 END AS NR_PONT_ORC_FUT,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
                WHEN CD_FAIXA_ORC = 2 THEN 2 WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
                ELSE 0 END AS NR_PONT_ORC_OBR,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END AS NR_PONT_PRFL_IND,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0
                          OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
                ELSE CASE WHEN CD_MAC_PRFL_CLI = 1 THEN 0 ELSE 1 END END AS NR_PONT_PRFL_ESS,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0
                          OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
                ELSE CASE WHEN CD_MAC_PRFL_CLI = 1 THEN 1 ELSE 0 END END AS NR_PONT_PRFL_NAO_ESS,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0
                          OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
                ELSE CASE WHEN CD_MAC_PRFL_CLI = 3 THEN 2 WHEN CD_MAC_PRFL_CLI = 2 THEN 1 ELSE 0 END END AS NR_PONT_PRFL_FUT,
           CASE WHEN QT_TRANS_TOTAL IS NULL OR QT_TRANS_TOTAL = 0
                          OR CD_MAC_PRFL_CLI IS NULL OR CD_MAC_PRFL_CLI NOT IN (1, 2, 3) THEN NULL
                ELSE CASE WHEN CD_MAC_PRFL_CLI = 1 THEN 2 ELSE 0 END END AS NR_PONT_PRFL_OBR
    FROM faixas
), finais AS (
    SELECT *,
           NR_PONT_CONC_IND AS NR_PONT_IND_FIM,
           NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS AS NR_PONT_ESS_FIM,
           NR_PONT_CONC_NAO_ESS + NR_PONT_ORC_NAO_ESS + NR_PONT_PRFL_NAO_ESS AS NR_PONT_NAO_ESS_FIM,
           NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT AS NR_PONT_FUT_FIM,
           NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR AS NR_PONT_OBR_FIM
    FROM pontos
), classificacao AS (
    SELECT *,
           ARRAY(NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM) AS PONTUACOES,
           CASE WHEN NR_PONT_IND_FIM IS NOT NULL
                      AND NR_PONT_ESS_FIM IS NOT NULL
                      AND NR_PONT_NAO_ESS_FIM IS NOT NULL
                      AND NR_PONT_FUT_FIM IS NOT NULL
                      AND NR_PONT_OBR_FIM IS NOT NULL
                THEN 'S' ELSE 'N' END AS FL_PONTUACAO_COMPLETA
    FROM finais
), ranking AS (
    SELECT *,
           CASE WHEN FL_PONTUACAO_COMPLETA = 'S' THEN ARRAY_MAX(PONTUACOES) END AS NR_PONT_MAX,
           CASE WHEN FL_PONTUACAO_COMPLETA = 'S'
                THEN SIZE(FILTER(PONTUACOES, p -> p = ARRAY_MAX(PONTUACOES))) END AS QT_TEMAS_PONT_MAX
    FROM classificacao
), vencedor AS (
    SELECT *,
           CASE WHEN FL_PONTUACAO_COMPLETA <> 'S' THEN NULL
                WHEN QT_TEMAS_PONT_MAX > 1 THEN 9
                ELSE INT(ARRAY_POSITION(PONTUACOES, NR_PONT_MAX)) END AS CD_TEMA_VENCEDOR
    FROM ranking
)
SELECT CD_CLI, cenario AS CENARIO,
       VL_RES_ORC, PC_SAI_ENT,
       CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 0 THEN 0 WHEN CD_FAIXA_ORC IN (1, 2) THEN 2 ELSE 1 END AS CD_RES_ORC,
       CASE WHEN CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
            WHEN CD_FAIXA_ORC IN (1, 2) THEN 'Deficitário' ELSE 'Superavitário' END AS TX_RES_ORC,
       CD_FAIXA_ORC,
       CASE WHEN CD_FAIXA_ORC IN (1, 3) THEN 'Moderado'
            WHEN CD_FAIXA_ORC IN (2, 4) THEN 'Acentuado' END AS TX_STS_RES,
       CASE CD_FAIXA_ORC WHEN 0 THEN 'Neutro' WHEN 1 THEN 'Deficitário Moderado'
                        WHEN 2 THEN 'Deficitário Acentuado' WHEN 3 THEN 'Superavitário Moderado'
                        WHEN 4 THEN 'Superavitário Acentuado' END AS TX_STS_FINAL,
       PC_SAI_IND, PC_SAI_ESS, PC_SAI_NAO_ESS, PC_SAI_FUT, PC_SAI_OBR,
       CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
       CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
       CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
       CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
       CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR,
       NR_PONT_CONC_IND, NR_PONT_CONC_ESS, NR_PONT_CONC_NAO_ESS, NR_PONT_CONC_FUT, NR_PONT_CONC_OBR,
       NR_PONT_ORC_IND, NR_PONT_ORC_ESS, NR_PONT_ORC_NAO_ESS, NR_PONT_ORC_FUT, NR_PONT_ORC_OBR,
       NR_PONT_PRFL_IND, NR_PONT_PRFL_ESS, NR_PONT_PRFL_NAO_ESS, NR_PONT_PRFL_FUT, NR_PONT_PRFL_OBR,
       NR_PONT_IND_FIM, NR_PONT_ESS_FIM, NR_PONT_NAO_ESS_FIM, NR_PONT_FUT_FIM, NR_PONT_OBR_FIM,
       FL_PONTUACAO_COMPLETA, NR_PONT_MAX, QT_TEMAS_PONT_MAX, CD_TEMA_VENCEDOR,
       CASE CD_TEMA_VENCEDOR
           WHEN 1 THEN 'Categorização dos Gastos'
           WHEN 2 THEN 'Gestão de Orçamento'
           WHEN 3 THEN 'Consumo Planejado'
           WHEN 4 THEN 'Formação de Reserva'
           WHEN 5 THEN 'Uso Consciente do Crédito'
           WHEN 9 THEN 'Empate'
       END AS TX_TEMA_VENCEDOR
FROM vencedor
""")
spark.sql("CACHE TABLE rts_motor")
spark.sql("SELECT CENARIO, COUNT(*) AS QT_CLIENTES FROM rts_motor GROUP BY CENARIO").show()


In [ ]:
%%time
%%spark
resultado_validado = False
# Projetar as 142 colunas oficiais na ordem e nos tipos contratados.
spark.sql("""
CREATE OR REPLACE TEMP VIEW rts_resultado AS
SELECT
       CAST(b.CD_CLI AS INT) AS CD_CLI,
       CAST(b.DT_EXEA AS DATE) AS DT_EXEA,
       CAST(b.DT_MES_EXEA AS DATE) AS DT_MES_EXEA,
       CAST(b.TS_INCL_TRAN_REF AS TIMESTAMP) AS TS_INCL_TRAN_REF,
       CAST(b.FL_CPF_UNICO AS STRING) AS FL_CPF_UNICO,
       CAST(b.CD_CPF AS DECIMAL(14,0)) AS CD_CPF,
       CAST(b.FL_CONTA_ELEGIVEL_UNICA AS STRING) AS FL_CONTA_ELEGIVEL_UNICA,
       CAST(b.TS_DD_INC_MM_CLC_BLC_REF AS TIMESTAMP) AS TS_DD_INC_MM_CLC_BLC_REF,
       CAST(b.DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
       CAST(b.DD_INC_MM_CLC_BLC_FALLBACK AS SMALLINT) AS DD_INC_MM_CLC_BLC_FALLBACK,
       CAST(b.DT_REN_PRES_REF AS DATE) AS DT_REN_PRES_REF,
       CAST(b.VL_REN_PRES AS DECIMAL(17,2)) AS VL_REN_PRES,
       CAST(b.DT_REF_PRFL AS DATE) AS DT_REF_PRFL,
       CAST(b.CD_MAC_PRFL_CLI AS INT) AS CD_MAC_PRFL_CLI,
       CAST(b.NM_MAC_PRFL_CLI AS STRING) AS NM_MAC_PRFL_CLI,
       CAST(b.CD_MIC_PRFL_CLI AS INT) AS CD_MIC_PRFL_CLI,
       CAST(b.NM_MIC_PRFL_CLI AS STRING) AS NM_MIC_PRFL_CLI,
       CAST(b.DT_REF_INI AS DATE) AS DT_REF_INI,
       CAST(b.DT_REF_FIM AS DATE) AS DT_REF_FIM,
       CAST(b.FL_SOMENTE_BRL AS STRING) AS FL_SOMENTE_BRL,
       CAST(b.FL_TEM_MOV_AGRO AS STRING) AS FL_TEM_MOV_AGRO,
       CAST(b.QT_TRANS_TOTAL AS BIGINT) AS QT_TRANS_TOTAL,
       CAST(b.QT_TRANS_ENT AS BIGINT) AS QT_TRANS_ENT,
       CAST(b.QT_TRANS_SAI AS BIGINT) AS QT_TRANS_SAI,
       CAST(b.VL_TRANS_ENT AS DECIMAL(25,2)) AS VL_TRANS_ENT,
       CAST(b.VL_TRANS_SAI AS DECIMAL(25,2)) AS VL_TRANS_SAI,
       CAST(b.VL_ENT_REN AS DECIMAL(25,2)) AS VL_ENT_REN,
       CAST(b.VL_ENT_EST AS DECIMAL(25,2)) AS VL_ENT_EST,
       CAST(b.VL_ENT_RESG AS DECIMAL(25,2)) AS VL_ENT_RESG,
       CAST(b.VL_ENT_OUT AS DECIMAL(25,2)) AS VL_ENT_OUT,
       CAST(b.VL_ENT_CRED AS DECIMAL(25,2)) AS VL_ENT_CRED,
       CAST(b.VL_ENT_TOTAL AS DECIMAL(25,2)) AS VL_ENT_TOTAL,
       CAST(b.VL_SAI_IND AS DECIMAL(25,2)) AS VL_SAI_IND,
       CAST(b.VL_SAI_ESS AS DECIMAL(25,2)) AS VL_SAI_ESS,
       CAST(b.VL_SAI_NAO_ESS AS DECIMAL(25,2)) AS VL_SAI_NAO_ESS,
       CAST(b.VL_SAI_FUT AS DECIMAL(25,2)) AS VL_SAI_FUT,
       CAST(b.VL_SAI_OBR AS DECIMAL(25,2)) AS VL_SAI_OBR,
       CAST(b.VL_SAI_TOTAL AS DECIMAL(25,2)) AS VL_SAI_TOTAL,
       CAST(o.VL_RES_ORC AS DECIMAL(25,2)) AS VL_RES_ORC,
       CAST(o.PC_SAI_ENT AS DECIMAL(9,6)) AS PC_SAI_ENT,
       CAST(o.CD_RES_ORC AS INT) AS CD_RES_ORC,
       CAST(o.TX_RES_ORC AS STRING) AS TX_RES_ORC,
       CAST(o.CD_FAIXA_ORC AS INT) AS CD_FAIXA_ORC,
       CAST(o.TX_STS_RES AS STRING) AS TX_STS_RES,
       CAST(o.TX_STS_FINAL AS STRING) AS TX_STS_FINAL,
       CAST(o.PC_SAI_IND AS DECIMAL(9,6)) AS PC_SAI_IND,
       CAST(o.PC_SAI_ESS AS DECIMAL(9,6)) AS PC_SAI_ESS,
       CAST(o.PC_SAI_NAO_ESS AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS,
       CAST(o.PC_SAI_FUT AS DECIMAL(9,6)) AS PC_SAI_FUT,
       CAST(o.PC_SAI_OBR AS DECIMAL(9,6)) AS PC_SAI_OBR,
       CAST(o.PC_REF_IND AS DECIMAL(9,6)) AS PC_REF_IND,
       CAST(o.PC_REF_ESS AS DECIMAL(9,6)) AS PC_REF_ESS,
       CAST(o.PC_REF_NAO_ESS AS DECIMAL(9,6)) AS PC_REF_NAO_ESS,
       CAST(o.PC_REF_FUT AS DECIMAL(9,6)) AS PC_REF_FUT,
       CAST(o.PC_REF_OBR AS DECIMAL(9,6)) AS PC_REF_OBR,
       CAST(o.NR_PONT_CONC_IND AS INT) AS NR_PONT_CONC_IND,
       CAST(o.NR_PONT_CONC_ESS AS INT) AS NR_PONT_CONC_ESS,
       CAST(o.NR_PONT_CONC_NAO_ESS AS INT) AS NR_PONT_CONC_NAO_ESS,
       CAST(o.NR_PONT_CONC_FUT AS INT) AS NR_PONT_CONC_FUT,
       CAST(o.NR_PONT_CONC_OBR AS INT) AS NR_PONT_CONC_OBR,
       CAST(o.NR_PONT_ORC_IND AS INT) AS NR_PONT_ORC_IND,
       CAST(o.NR_PONT_ORC_ESS AS INT) AS NR_PONT_ORC_ESS,
       CAST(o.NR_PONT_ORC_NAO_ESS AS INT) AS NR_PONT_ORC_NAO_ESS,
       CAST(o.NR_PONT_ORC_FUT AS INT) AS NR_PONT_ORC_FUT,
       CAST(o.NR_PONT_ORC_OBR AS INT) AS NR_PONT_ORC_OBR,
       CAST(o.NR_PONT_PRFL_IND AS INT) AS NR_PONT_PRFL_IND,
       CAST(o.NR_PONT_PRFL_ESS AS INT) AS NR_PONT_PRFL_ESS,
       CAST(o.NR_PONT_PRFL_NAO_ESS AS INT) AS NR_PONT_PRFL_NAO_ESS,
       CAST(o.NR_PONT_PRFL_FUT AS INT) AS NR_PONT_PRFL_FUT,
       CAST(o.NR_PONT_PRFL_OBR AS INT) AS NR_PONT_PRFL_OBR,
       CAST(o.NR_PONT_IND_FIM AS INT) AS NR_PONT_IND_FIM,
       CAST(o.NR_PONT_ESS_FIM AS INT) AS NR_PONT_ESS_FIM,
       CAST(o.NR_PONT_NAO_ESS_FIM AS INT) AS NR_PONT_NAO_ESS_FIM,
       CAST(o.NR_PONT_FUT_FIM AS INT) AS NR_PONT_FUT_FIM,
       CAST(o.NR_PONT_OBR_FIM AS INT) AS NR_PONT_OBR_FIM,
       CAST(o.FL_PONTUACAO_COMPLETA AS STRING) AS FL_PONTUACAO_COMPLETA,
       CAST(o.NR_PONT_MAX AS INT) AS NR_PONT_MAX,
       CAST(o.QT_TEMAS_PONT_MAX AS INT) AS QT_TEMAS_PONT_MAX,
       CAST(o.CD_TEMA_VENCEDOR AS INT) AS CD_TEMA_VENCEDOR,
       CAST(o.TX_TEMA_VENCEDOR AS STRING) AS TX_TEMA_VENCEDOR,
       CAST(r.VL_RES_ORC AS DECIMAL(25,2)) AS VL_RES_ORC_RENDA_PRESUMIDA,
       CAST(r.PC_SAI_ENT AS DECIMAL(9,6)) AS PC_SAI_ENT_RENDA_PRESUMIDA,
       CAST(r.CD_RES_ORC AS INT) AS CD_RES_ORC_RENDA_PRESUMIDA,
       CAST(r.TX_RES_ORC AS STRING) AS TX_RES_ORC_RENDA_PRESUMIDA,
       CAST(r.CD_FAIXA_ORC AS INT) AS CD_FAIXA_ORC_RENDA_PRESUMIDA,
       CAST(r.TX_STS_RES AS STRING) AS TX_STS_RES_RENDA_PRESUMIDA,
       CAST(r.TX_STS_FINAL AS STRING) AS TX_STS_FINAL_RENDA_PRESUMIDA,
       CAST(r.PC_SAI_IND AS DECIMAL(9,6)) AS PC_SAI_IND_RENDA_PRESUMIDA,
       CAST(r.PC_SAI_ESS AS DECIMAL(9,6)) AS PC_SAI_ESS_RENDA_PRESUMIDA,
       CAST(r.PC_SAI_NAO_ESS AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS_RENDA_PRESUMIDA,
       CAST(r.PC_SAI_FUT AS DECIMAL(9,6)) AS PC_SAI_FUT_RENDA_PRESUMIDA,
       CAST(r.PC_SAI_OBR AS DECIMAL(9,6)) AS PC_SAI_OBR_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_CONC_IND AS INT) AS NR_PONT_CONC_IND_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_CONC_ESS AS INT) AS NR_PONT_CONC_ESS_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_CONC_NAO_ESS AS INT) AS NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_CONC_FUT AS INT) AS NR_PONT_CONC_FUT_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_CONC_OBR AS INT) AS NR_PONT_CONC_OBR_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_ORC_ESS AS INT) AS NR_PONT_ORC_ESS_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_ORC_NAO_ESS AS INT) AS NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_ORC_FUT AS INT) AS NR_PONT_ORC_FUT_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_ORC_OBR AS INT) AS NR_PONT_ORC_OBR_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_IND_FIM AS INT) AS NR_PONT_IND_FIM_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_ESS_FIM AS INT) AS NR_PONT_ESS_FIM_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_NAO_ESS_FIM AS INT) AS NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_FUT_FIM AS INT) AS NR_PONT_FUT_FIM_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_OBR_FIM AS INT) AS NR_PONT_OBR_FIM_RENDA_PRESUMIDA,
       CAST(r.FL_PONTUACAO_COMPLETA AS STRING) AS FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA,
       CAST(r.NR_PONT_MAX AS INT) AS NR_PONT_MAX_RENDA_PRESUMIDA,
       CAST(r.QT_TEMAS_PONT_MAX AS INT) AS QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA,
       CAST(r.CD_TEMA_VENCEDOR AS INT) AS CD_TEMA_VENCEDOR_RENDA_PRESUMIDA,
       CAST(r.TX_TEMA_VENCEDOR AS STRING) AS TX_TEMA_VENCEDOR_RENDA_PRESUMIDA,
       CAST(e.VL_RES_ORC AS DECIMAL(25,2)) AS VL_RES_ORC_ENTRADAS_REALIZADAS,
       CAST(e.PC_SAI_ENT AS DECIMAL(9,6)) AS PC_SAI_ENT_ENTRADAS_REALIZADAS,
       CAST(e.CD_RES_ORC AS INT) AS CD_RES_ORC_ENTRADAS_REALIZADAS,
       CAST(e.TX_RES_ORC AS STRING) AS TX_RES_ORC_ENTRADAS_REALIZADAS,
       CAST(e.CD_FAIXA_ORC AS INT) AS CD_FAIXA_ORC_ENTRADAS_REALIZADAS,
       CAST(e.TX_STS_RES AS STRING) AS TX_STS_RES_ENTRADAS_REALIZADAS,
       CAST(e.TX_STS_FINAL AS STRING) AS TX_STS_FINAL_ENTRADAS_REALIZADAS,
       CAST(e.PC_SAI_IND AS DECIMAL(9,6)) AS PC_SAI_IND_ENTRADAS_REALIZADAS,
       CAST(e.PC_SAI_ESS AS DECIMAL(9,6)) AS PC_SAI_ESS_ENTRADAS_REALIZADAS,
       CAST(e.PC_SAI_NAO_ESS AS DECIMAL(9,6)) AS PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS,
       CAST(e.PC_SAI_FUT AS DECIMAL(9,6)) AS PC_SAI_FUT_ENTRADAS_REALIZADAS,
       CAST(e.PC_SAI_OBR AS DECIMAL(9,6)) AS PC_SAI_OBR_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_CONC_IND AS INT) AS NR_PONT_CONC_IND_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_CONC_ESS AS INT) AS NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_CONC_NAO_ESS AS INT) AS NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_CONC_FUT AS INT) AS NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_CONC_OBR AS INT) AS NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_ORC_ESS AS INT) AS NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_ORC_NAO_ESS AS INT) AS NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_ORC_FUT AS INT) AS NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_ORC_OBR AS INT) AS NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_IND_FIM AS INT) AS NR_PONT_IND_FIM_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_ESS_FIM AS INT) AS NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_NAO_ESS_FIM AS INT) AS NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_FUT_FIM AS INT) AS NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_OBR_FIM AS INT) AS NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS,
       CAST(e.FL_PONTUACAO_COMPLETA AS STRING) AS FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS,
       CAST(e.NR_PONT_MAX AS INT) AS NR_PONT_MAX_ENTRADAS_REALIZADAS,
       CAST(e.QT_TEMAS_PONT_MAX AS INT) AS QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS,
       CAST(e.CD_TEMA_VENCEDOR AS INT) AS CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS,
       CAST(e.TX_TEMA_VENCEDOR AS STRING) AS TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS
FROM rts_base b
LEFT JOIN rts_motor o ON b.CD_CLI = o.CD_CLI AND o.CENARIO = 'OFICIAL'
LEFT JOIN rts_motor r ON b.CD_CLI = r.CD_CLI AND r.CENARIO = 'RENDA_PRESUMIDA'
LEFT JOIN rts_motor e ON b.CD_CLI = e.CD_CLI AND e.CENARIO = 'ENTRADAS_REALIZADAS'
""")
spark.sql("CACHE TABLE rts_resultado")
df_resultado = spark.table('rts_resultado')
spark.sql("UNCACHE TABLE rts_motor")
spark.sql("UNCACHE TABLE rts_base")


## 6. Validações


In [ ]:
%%time
%%spark
# Validar o contrato e a permanência dos clientes antes da publicação.
contrato_esperado = [
    ('CD_CLI', 'int'),
    ('DT_EXEA', 'date'),
    ('DT_MES_EXEA', 'date'),
    ('TS_INCL_TRAN_REF', 'timestamp'),
    ('FL_CPF_UNICO', 'string'),
    ('CD_CPF', 'decimal(14,0)'),
    ('FL_CONTA_ELEGIVEL_UNICA', 'string'),
    ('TS_DD_INC_MM_CLC_BLC_REF', 'timestamp'),
    ('DD_INC_MM_CLC_BLC', 'smallint'),
    ('DD_INC_MM_CLC_BLC_FALLBACK', 'smallint'),
    ('DT_REN_PRES_REF', 'date'),
    ('VL_REN_PRES', 'decimal(17,2)'),
    ('DT_REF_PRFL', 'date'),
    ('CD_MAC_PRFL_CLI', 'int'),
    ('NM_MAC_PRFL_CLI', 'string'),
    ('CD_MIC_PRFL_CLI', 'int'),
    ('NM_MIC_PRFL_CLI', 'string'),
    ('DT_REF_INI', 'date'),
    ('DT_REF_FIM', 'date'),
    ('FL_SOMENTE_BRL', 'string'),
    ('FL_TEM_MOV_AGRO', 'string'),
    ('QT_TRANS_TOTAL', 'bigint'),
    ('QT_TRANS_ENT', 'bigint'),
    ('QT_TRANS_SAI', 'bigint'),
    ('VL_TRANS_ENT', 'decimal(25,2)'),
    ('VL_TRANS_SAI', 'decimal(25,2)'),
    ('VL_ENT_REN', 'decimal(25,2)'),
    ('VL_ENT_EST', 'decimal(25,2)'),
    ('VL_ENT_RESG', 'decimal(25,2)'),
    ('VL_ENT_OUT', 'decimal(25,2)'),
    ('VL_ENT_CRED', 'decimal(25,2)'),
    ('VL_ENT_TOTAL', 'decimal(25,2)'),
    ('VL_SAI_IND', 'decimal(25,2)'),
    ('VL_SAI_ESS', 'decimal(25,2)'),
    ('VL_SAI_NAO_ESS', 'decimal(25,2)'),
    ('VL_SAI_FUT', 'decimal(25,2)'),
    ('VL_SAI_OBR', 'decimal(25,2)'),
    ('VL_SAI_TOTAL', 'decimal(25,2)'),
    ('VL_RES_ORC', 'decimal(25,2)'),
    ('PC_SAI_ENT', 'decimal(9,6)'),
    ('CD_RES_ORC', 'int'),
    ('TX_RES_ORC', 'string'),
    ('CD_FAIXA_ORC', 'int'),
    ('TX_STS_RES', 'string'),
    ('TX_STS_FINAL', 'string'),
    ('PC_SAI_IND', 'decimal(9,6)'),
    ('PC_SAI_ESS', 'decimal(9,6)'),
    ('PC_SAI_NAO_ESS', 'decimal(9,6)'),
    ('PC_SAI_FUT', 'decimal(9,6)'),
    ('PC_SAI_OBR', 'decimal(9,6)'),
    ('PC_REF_IND', 'decimal(9,6)'),
    ('PC_REF_ESS', 'decimal(9,6)'),
    ('PC_REF_NAO_ESS', 'decimal(9,6)'),
    ('PC_REF_FUT', 'decimal(9,6)'),
    ('PC_REF_OBR', 'decimal(9,6)'),
    ('NR_PONT_CONC_IND', 'int'),
    ('NR_PONT_CONC_ESS', 'int'),
    ('NR_PONT_CONC_NAO_ESS', 'int'),
    ('NR_PONT_CONC_FUT', 'int'),
    ('NR_PONT_CONC_OBR', 'int'),
    ('NR_PONT_ORC_IND', 'int'),
    ('NR_PONT_ORC_ESS', 'int'),
    ('NR_PONT_ORC_NAO_ESS', 'int'),
    ('NR_PONT_ORC_FUT', 'int'),
    ('NR_PONT_ORC_OBR', 'int'),
    ('NR_PONT_PRFL_IND', 'int'),
    ('NR_PONT_PRFL_ESS', 'int'),
    ('NR_PONT_PRFL_NAO_ESS', 'int'),
    ('NR_PONT_PRFL_FUT', 'int'),
    ('NR_PONT_PRFL_OBR', 'int'),
    ('NR_PONT_IND_FIM', 'int'),
    ('NR_PONT_ESS_FIM', 'int'),
    ('NR_PONT_NAO_ESS_FIM', 'int'),
    ('NR_PONT_FUT_FIM', 'int'),
    ('NR_PONT_OBR_FIM', 'int'),
    ('FL_PONTUACAO_COMPLETA', 'string'),
    ('NR_PONT_MAX', 'int'),
    ('QT_TEMAS_PONT_MAX', 'int'),
    ('CD_TEMA_VENCEDOR', 'int'),
    ('TX_TEMA_VENCEDOR', 'string'),
    ('VL_RES_ORC_RENDA_PRESUMIDA', 'decimal(25,2)'),
    ('PC_SAI_ENT_RENDA_PRESUMIDA', 'decimal(9,6)'),
    ('CD_RES_ORC_RENDA_PRESUMIDA', 'int'),
    ('TX_RES_ORC_RENDA_PRESUMIDA', 'string'),
    ('CD_FAIXA_ORC_RENDA_PRESUMIDA', 'int'),
    ('TX_STS_RES_RENDA_PRESUMIDA', 'string'),
    ('TX_STS_FINAL_RENDA_PRESUMIDA', 'string'),
    ('PC_SAI_IND_RENDA_PRESUMIDA', 'decimal(9,6)'),
    ('PC_SAI_ESS_RENDA_PRESUMIDA', 'decimal(9,6)'),
    ('PC_SAI_NAO_ESS_RENDA_PRESUMIDA', 'decimal(9,6)'),
    ('PC_SAI_FUT_RENDA_PRESUMIDA', 'decimal(9,6)'),
    ('PC_SAI_OBR_RENDA_PRESUMIDA', 'decimal(9,6)'),
    ('NR_PONT_CONC_IND_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_CONC_ESS_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_CONC_FUT_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_CONC_OBR_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_ORC_ESS_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_ORC_FUT_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_ORC_OBR_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_IND_FIM_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_ESS_FIM_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_FUT_FIM_RENDA_PRESUMIDA', 'int'),
    ('NR_PONT_OBR_FIM_RENDA_PRESUMIDA', 'int'),
    ('FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA', 'string'),
    ('NR_PONT_MAX_RENDA_PRESUMIDA', 'int'),
    ('QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA', 'int'),
    ('CD_TEMA_VENCEDOR_RENDA_PRESUMIDA', 'int'),
    ('TX_TEMA_VENCEDOR_RENDA_PRESUMIDA', 'string'),
    ('VL_RES_ORC_ENTRADAS_REALIZADAS', 'decimal(25,2)'),
    ('PC_SAI_ENT_ENTRADAS_REALIZADAS', 'decimal(9,6)'),
    ('CD_RES_ORC_ENTRADAS_REALIZADAS', 'int'),
    ('TX_RES_ORC_ENTRADAS_REALIZADAS', 'string'),
    ('CD_FAIXA_ORC_ENTRADAS_REALIZADAS', 'int'),
    ('TX_STS_RES_ENTRADAS_REALIZADAS', 'string'),
    ('TX_STS_FINAL_ENTRADAS_REALIZADAS', 'string'),
    ('PC_SAI_IND_ENTRADAS_REALIZADAS', 'decimal(9,6)'),
    ('PC_SAI_ESS_ENTRADAS_REALIZADAS', 'decimal(9,6)'),
    ('PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS', 'decimal(9,6)'),
    ('PC_SAI_FUT_ENTRADAS_REALIZADAS', 'decimal(9,6)'),
    ('PC_SAI_OBR_ENTRADAS_REALIZADAS', 'decimal(9,6)'),
    ('NR_PONT_CONC_IND_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_IND_FIM_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS', 'int'),
    ('NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS', 'int'),
    ('FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS', 'string'),
    ('NR_PONT_MAX_ENTRADAS_REALIZADAS', 'int'),
    ('QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS', 'int'),
    ('CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS', 'int'),
    ('TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS', 'string'),
]
assert len(df_resultado.columns) == 142, 'Resultado deve possuir 142 colunas.'
assert df_resultado.dtypes == contrato_esperado, 'Nomes, ordem ou tipos divergentes do contrato.'
metricas_resultado = spark.sql("""
SELECT COUNT(*) AS QT_RESULTADO,
       ASSERT_TRUE(COUNT(*) > 0, 'Resultado vazio.'),
       ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT r.CD_CLI), 'Cliente nulo ou duplicado no resultado.'),
       ASSERT_TRUE(COUNT(CASE WHEN p.CD_CLI IS NULL OR r.CD_CLI IS NULL THEN 1 END) = 0,
                   'Resultado não preservou os clientes do público.'),
       ASSERT_TRUE(COUNT(CASE WHEN r.FL_CPF_UNICO = 'N' AND r.CD_CPF IS NOT NULL THEN 1 END) = 0,
                   'CPF sem unicidade deve ser NULL.'),
       ASSERT_TRUE(COUNT(CASE WHEN r.DD_INC_MM_CLC_BLC_FALLBACK IS NULL
                                  AND (r.DT_REF_INI IS NOT NULL OR r.DT_REF_FIM IS NOT NULL)
                             THEN 1 END) = 0, 'Janela disponível sem ciclo.'),
       ASSERT_TRUE(COUNT(CASE WHEN r.QT_TRANS_TOTAL IS NULL
                                  OR (r.QT_TRANS_TOTAL = 0 AND (
                                      r.QT_TRANS_ENT IS NOT NULL OR r.QT_TRANS_SAI IS NOT NULL
                                      OR r.FL_TEM_MOV_AGRO IS NOT NULL
                                      OR NOT (r.VL_TRANS_ENT <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_TRANS_SAI <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_ENT_REN <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_ENT_EST <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_ENT_RESG <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_ENT_OUT <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_ENT_CRED <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_ENT_TOTAL <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_SAI_IND <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_SAI_ESS <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_SAI_NAO_ESS <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_SAI_FUT <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_SAI_OBR <=> CAST(0 AS DECIMAL(25,2)))
                                      OR NOT (r.VL_SAI_TOTAL <=> CAST(0 AS DECIMAL(25,2)))
                                  )) THEN 1 END) = 0, 'Sem BRL, preservar NULL nas quantidades parciais e zero nos valores.'),
       ASSERT_TRUE(COUNT(CASE WHEN r.VL_REN_PRES IS NULL AND (
                                      r.PC_SAI_IND IS NOT NULL OR r.PC_SAI_ESS IS NOT NULL
                                      OR r.PC_SAI_NAO_ESS IS NOT NULL OR r.PC_SAI_FUT IS NOT NULL
                                      OR r.PC_SAI_OBR IS NOT NULL)
                             THEN 1 END) = 0, 'Percentuais oficiais disponíveis sem renda.'),
       ASSERT_TRUE(COUNT(CASE WHEN (r.DT_REF_INI IS NULL OR r.DT_REF_FIM IS NULL) AND (
                                      r.VL_RES_ORC_RENDA_PRESUMIDA IS NOT NULL
                                      OR r.VL_RES_ORC_ENTRADAS_REALIZADAS IS NOT NULL
                                      OR NOT (r.FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA <=> 'N')
                                      OR NOT (r.FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS <=> 'N'))
                             THEN 1 END) = 0, 'Cenários calculados sem janela.')
FROM rts_resultado r
FULL OUTER JOIN rts_publico p ON r.CD_CLI = p.CD_CLI
""").first()
resultado_validado = True
print(f'Resultado validado: {metricas_resultado["QT_RESULTADO"]} clientes, 142 colunas.')


## 7. Publicação

A publicação substitui integralmente a fotografia anterior: `DROP → CREATE → INSERT → readback`, conforme decisão explícita deste pipeline. Execute após as validações.


In [ ]:
%%time
%%spark
assert resultado_validado and metricas_resultado['QT_RESULTADO'] > 0, 'Publicação bloqueada: valide o resultado.'
assert df_resultado.dtypes == contrato_esperado, 'Contrato alterado após validação.'
print(f'Destino: {spark.catalog.currentDatabase()}.ANA_RADAR_FIN_CLI')

# Remover a fotografia anterior após a validação do resultado.
spark.sql("DROP TABLE IF EXISTS ANA_RADAR_FIN_CLI")

# Recriar a tabela com o DDL oficial completo.
spark.sql("""
CREATE TABLE ANA_RADAR_FIN_CLI (

    -- ============================================================
    -- IDENTIFICAÇÃO, REFERÊNCIAS E CONTEXTO
    -- ============================================================

    CD_CLI INT
        COMMENT 'Código identificador do cliente processado pelo Radar',

    DT_EXEA DATE
        COMMENT 'Data de execução do cálculo do Radar',

    DT_MES_EXEA DATE
        COMMENT 'Primeiro dia do mês correspondente à data de execução',

    TS_INCL_TRAN_REF TIMESTAMP
        COMMENT 'Maior timestamp de inclusão de transação utilizado como referência para formação da janela financeira',

    FL_CPF_UNICO STRING
        COMMENT 'Indica se existe exatamente um CPF distinto associado ao cliente no público analisado: S=sim; N=não',

    CD_CPF DECIMAL(14,0)
        COMMENT 'Código do CPF único associado ao cliente; nulo quando não existe unicidade',

    FL_CONTA_ELEGIVEL_UNICA STRING
        COMMENT 'Indica se existe exatamente uma conta elegível para determinação do ciclo financeiro: S=sim; N=não',

    TS_DD_INC_MM_CLC_BLC_REF TIMESTAMP
        COMMENT 'Timestamp de referência do registro de ciclo financeiro selecionado',

    DD_INC_MM_CLC_BLC SMALLINT
        COMMENT 'Dia de início do ciclo financeiro obtido da fonte',

    DD_INC_MM_CLC_BLC_FALLBACK SMALLINT
        COMMENT 'Dia de ciclo efetivamente utilizado após aplicação da regra de fallback',

    DT_REN_PRES_REF DATE
        COMMENT 'Data de referência da renda presumida selecionada',

    VL_REN_PRES DECIMAL(17,2)
        COMMENT 'Valor da renda presumida utilizada pelo Radar',

    DT_REF_PRFL DATE
        COMMENT 'Data de referência do perfil financeiro selecionado',

    CD_MAC_PRFL_CLI INT
        COMMENT 'Código do macroperfil financeiro recebido da fonte; o notebook usa 1, 2 e 3 na pontuação, sem definir rótulos negociais próprios',

    NM_MAC_PRFL_CLI STRING
        COMMENT 'Nome do macroperfil financeiro recebido da fonte',

    CD_MIC_PRFL_CLI INT
        COMMENT 'Código do microperfil financeiro recebido da fonte; o motor atual não cria um de-para negocial próprio para este código',

    NM_MIC_PRFL_CLI STRING
        COMMENT 'Nome do microperfil financeiro recebido da fonte',

    DT_REF_INI DATE
        COMMENT 'Data inicial da janela financeira analisada',

    DT_REF_FIM DATE
        COMMENT 'Data final da janela financeira analisada',

    FL_SOMENTE_BRL STRING
        COMMENT 'Calculado sobre os movimentos efetivos: S quando a única moeda distinta é BRL; N nos demais casos; NULL quando não há movimentos efetivos',

    FL_TEM_MOV_AGRO STRING
        COMMENT 'Indica movimentação agro entre os movimentos efetivos BRL: S=possui movimento agro; N=não possui; NULL=sem movimentos BRL',


    -- ============================================================
    -- MOVIMENTAÇÃO
    -- ============================================================

    QT_TRANS_TOTAL BIGINT
        COMMENT 'Quantidade total de movimentos efetivos BRL considerados na janela',

    QT_TRANS_ENT BIGINT
        COMMENT 'Quantidade de movimentos efetivos de entrada em BRL',

    QT_TRANS_SAI BIGINT
        COMMENT 'Quantidade de movimentos efetivos de saída em BRL',

    VL_TRANS_ENT DECIMAL(25,2)
        COMMENT 'Valor das entradas participantes do cálculo; no contrato atual corresponde ao VL_ENT_TOTAL',

    VL_TRANS_SAI DECIMAL(25,2)
        COMMENT 'Valor das saídas participantes do cálculo; no contrato atual corresponde ao VL_SAI_TOTAL',


    -- ============================================================
    -- ENTRADAS
    -- ============================================================

    VL_ENT_REN DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Renda, classe 1',

    VL_ENT_EST DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Estorno, classe 2',

    VL_ENT_RESG DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Resgate, classe 3',

    VL_ENT_OUT DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Outras Entradas, classe 0',

    VL_ENT_CRED DECIMAL(25,2)
        COMMENT 'Valor das entradas classificadas como Crédito, classe 4',

    VL_ENT_TOTAL DECIMAL(25,2)
        COMMENT 'Valor total das entradas participantes do orçamento',


    -- ============================================================
    -- SAÍDAS
    -- ============================================================

    VL_SAI_IND DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Indeterminadas, classe 5',

    VL_SAI_ESS DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Essenciais, classe 6',

    VL_SAI_NAO_ESS DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Não Essenciais, classe 7',

    VL_SAI_FUT DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Futuro, classe 8',

    VL_SAI_OBR DECIMAL(25,2)
        COMMENT 'Valor das saídas classificadas como Obrigações, classe 9',

    VL_SAI_TOTAL DECIMAL(25,2)
        COMMENT 'Valor total das saídas participantes do orçamento',


    -- ============================================================
    -- RESULTADO ORÇAMENTÁRIO OFICIAL
    -- ============================================================

    VL_RES_ORC DECIMAL(25,2)
        COMMENT 'Resultado orçamentário calculado pela diferença entre VL_ENT_TOTAL e VL_SAI_TOTAL',

    PC_SAI_ENT DECIMAL(9,6)
        COMMENT 'Relação entre o total de saídas e a base de entrada utilizada no orçamento oficial',

    CD_RES_ORC INT
        COMMENT 'Código do resultado orçamentário: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC STRING
        COMMENT 'Descrição do resultado orçamentário correspondente ao CD_RES_ORC',

    CD_FAIXA_ORC INT
        COMMENT 'Código da faixa orçamentária: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES STRING
        COMMENT 'Intensidade do resultado orçamentário: Moderado ou Acentuado; NULL para resultado Neutro ou não calculado',

    TX_STS_FINAL STRING
        COMMENT 'Descrição final do resultado: Neutro, Deficitário Moderado, Deficitário Acentuado, Superavitário Moderado ou Superavitário Acentuado',


    -- ============================================================
    -- PERCENTUAIS OFICIAIS
    -- ============================================================

    PC_SAI_IND DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas em relação à renda presumida no resultado oficial',

    PC_SAI_ESS DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais em relação à renda presumida no resultado oficial',

    PC_SAI_NAO_ESS DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais em relação à renda presumida no resultado oficial',

    PC_SAI_FUT DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro em relação à renda presumida no resultado oficial',

    PC_SAI_OBR DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações em relação à renda presumida no resultado oficial',

    PC_REF_IND DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Indeterminada: 0.750000 corresponde a 75%',

    PC_REF_ESS DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Essencial: 0.500000 corresponde a 50%',

    PC_REF_NAO_ESS DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Não Essencial: 0.300000 corresponde a 30%',

    PC_REF_FUT DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Futuro: 0.200000 corresponde a 20%',

    PC_REF_OBR DECIMAL(9,6)
        COMMENT 'Percentual de referência da categoria Obrigações: 0.300000 corresponde a 30%',


    -- ============================================================
    -- PONTUAÇÃO DE CONCENTRAÇÃO
    -- ============================================================

    NR_PONT_CONC_IND INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada; valores produzidos pelo motor incluem 0 e 99',

    NR_PONT_CONC_ESS INT
        COMMENT 'Pontuação de concentração da categoria Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_NAO_ESS INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_FUT INT
        COMMENT 'Pontuação de concentração da categoria Futuro; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_CONC_OBR INT
        COMMENT 'Pontuação de concentração da categoria Obrigações; valores produzidos pelo motor: 0, 1 ou 2',


    -- ============================================================
    -- PONTUAÇÃO ORÇAMENTÁRIA
    -- ============================================================

    NR_PONT_ORC_IND INT
        COMMENT 'Pontuação orçamentária da categoria Indeterminada',

    NR_PONT_ORC_ESS INT
        COMMENT 'Pontuação orçamentária da categoria Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_NAO_ESS INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_FUT INT
        COMMENT 'Pontuação orçamentária da categoria Futuro; valores produzidos pelo motor: 0, 1 ou 2',

    NR_PONT_ORC_OBR INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações; valores produzidos pelo motor: 0, 1 ou 2',


    -- ============================================================
    -- PONTUAÇÃO DE PERFIL
    -- ============================================================

    NR_PONT_PRFL_IND INT
        COMMENT 'Pontuação de perfil da categoria Indeterminada',

    NR_PONT_PRFL_ESS INT
        COMMENT 'Pontuação de perfil da categoria Essencial',

    NR_PONT_PRFL_NAO_ESS INT
        COMMENT 'Pontuação de perfil da categoria Não Essencial',

    NR_PONT_PRFL_FUT INT
        COMMENT 'Pontuação de perfil da categoria Futuro',

    NR_PONT_PRFL_OBR INT
        COMMENT 'Pontuação de perfil da categoria Obrigações',


    -- ============================================================
    -- PONTUAÇÃO FINAL E TEMA VENCEDOR
    -- ============================================================

    NR_PONT_IND_FIM INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos',

    NR_PONT_ESS_FIM INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento',

    NR_PONT_NAO_ESS_FIM INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado',

    NR_PONT_FUT_FIM INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva',

    NR_PONT_OBR_FIM INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito',

    FL_PONTUACAO_COMPLETA STRING
        COMMENT 'Indica se as cinco pontuações finais foram calculadas: S=completa; N=incompleta',

    NR_PONT_MAX INT
        COMMENT 'Maior pontuação final obtida entre os cinco temas',

    QT_TEMAS_PONT_MAX INT
        COMMENT 'Quantidade de temas que possuem a maior pontuação final; exemplo: 1 indica vencedor único e valor maior que 1 indica empate',

    CD_TEMA_VENCEDOR INT
        COMMENT 'Código do tema vencedor: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR STRING
        COMMENT 'Descrição correspondente ao CD_TEMA_VENCEDOR',


    -- ============================================================
    -- CENÁRIO RENDA PRESUMIDA
    -- ============================================================

    VL_RES_ORC_RENDA_PRESUMIDA DECIMAL(25,2)
        COMMENT 'Resultado orçamentário recalculado no cenário RENDA_PRESUMIDA',

    PC_SAI_ENT_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Relação entre saídas e base utilizada no cenário RENDA_PRESUMIDA',

    CD_RES_ORC_RENDA_PRESUMIDA INT
        COMMENT 'Código do resultado orçamentário no cenário RENDA_PRESUMIDA: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição do resultado orçamentário no cenário RENDA_PRESUMIDA',

    CD_FAIXA_ORC_RENDA_PRESUMIDA INT
        COMMENT 'Código da faixa orçamentária no cenário RENDA_PRESUMIDA: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES_RENDA_PRESUMIDA STRING
        COMMENT 'Intensidade do resultado orçamentário no cenário RENDA_PRESUMIDA',

    TX_STS_FINAL_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição final do resultado orçamentário no cenário RENDA_PRESUMIDA',

    PC_SAI_IND_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas no cenário RENDA_PRESUMIDA',

    PC_SAI_ESS_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais no cenário RENDA_PRESUMIDA',

    PC_SAI_NAO_ESS_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais no cenário RENDA_PRESUMIDA',

    PC_SAI_FUT_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro no cenário RENDA_PRESUMIDA',

    PC_SAI_OBR_RENDA_PRESUMIDA DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_IND_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_NAO_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_FUT_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Futuro no cenário RENDA_PRESUMIDA',

    NR_PONT_CONC_OBR_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação de concentração da categoria Obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_NAO_ESS_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_FUT_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Futuro no cenário RENDA_PRESUMIDA',

    NR_PONT_ORC_OBR_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações no cenário RENDA_PRESUMIDA',

    NR_PONT_IND_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos, no cenário RENDA_PRESUMIDA',

    NR_PONT_ESS_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento, no cenário RENDA_PRESUMIDA',

    NR_PONT_NAO_ESS_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado, no cenário RENDA_PRESUMIDA',

    NR_PONT_FUT_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva, no cenário RENDA_PRESUMIDA',

    NR_PONT_OBR_FIM_RENDA_PRESUMIDA INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito, no cenário RENDA_PRESUMIDA',

    FL_PONTUACAO_COMPLETA_RENDA_PRESUMIDA STRING
        COMMENT 'Indica se as cinco pontuações foram calculadas no cenário RENDA_PRESUMIDA: S=completa; N=incompleta',

    NR_PONT_MAX_RENDA_PRESUMIDA INT
        COMMENT 'Maior pontuação final obtida no cenário RENDA_PRESUMIDA',

    QT_TEMAS_PONT_MAX_RENDA_PRESUMIDA INT
        COMMENT 'Quantidade de temas com a maior pontuação no cenário RENDA_PRESUMIDA',

    CD_TEMA_VENCEDOR_RENDA_PRESUMIDA INT
        COMMENT 'Código do tema vencedor no cenário RENDA_PRESUMIDA: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR_RENDA_PRESUMIDA STRING
        COMMENT 'Descrição correspondente ao tema vencedor no cenário RENDA_PRESUMIDA',


    -- ============================================================
    -- CENÁRIO ENTRADAS REALIZADAS
    -- ============================================================

    VL_RES_ORC_ENTRADAS_REALIZADAS DECIMAL(25,2)
        COMMENT 'Resultado orçamentário recalculado no cenário ENTRADAS_REALIZADAS',

    PC_SAI_ENT_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Relação entre saídas e base utilizada no cenário ENTRADAS_REALIZADAS',

    CD_RES_ORC_ENTRADAS_REALIZADAS INT
        COMMENT 'Código do resultado orçamentário no cenário ENTRADAS_REALIZADAS: 0=Neutro; 1=Superavitário; 2=Deficitário',

    TX_RES_ORC_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    CD_FAIXA_ORC_ENTRADAS_REALIZADAS INT
        COMMENT 'Código da faixa orçamentária no cenário ENTRADAS_REALIZADAS: 0=Neutro; 1=Deficitário Moderado; 2=Deficitário Acentuado; 3=Superavitário Moderado; 4=Superavitário Acentuado',

    TX_STS_RES_ENTRADAS_REALIZADAS STRING
        COMMENT 'Intensidade do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    TX_STS_FINAL_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição final do resultado orçamentário no cenário ENTRADAS_REALIZADAS',

    PC_SAI_IND_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas indeterminadas no cenário ENTRADAS_REALIZADAS',

    PC_SAI_ESS_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas essenciais no cenário ENTRADAS_REALIZADAS',

    PC_SAI_NAO_ESS_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas não essenciais no cenário ENTRADAS_REALIZADAS',

    PC_SAI_FUT_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas para futuro no cenário ENTRADAS_REALIZADAS',

    PC_SAI_OBR_ENTRADAS_REALIZADAS DECIMAL(9,6)
        COMMENT 'Proporção das saídas de obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_IND_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Indeterminada no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_NAO_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Não Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_FUT_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Futuro no cenário ENTRADAS_REALIZADAS',

    NR_PONT_CONC_OBR_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação de concentração da categoria Obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_NAO_ESS_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Não Essencial no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_FUT_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Futuro no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ORC_OBR_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação orçamentária da categoria Obrigações no cenário ENTRADAS_REALIZADAS',

    NR_PONT_IND_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 1, Categorização dos Gastos, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_ESS_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 2, Gestão de Orçamento, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_NAO_ESS_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 3, Consumo Planejado, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_FUT_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 4, Formação de Reserva, no cenário ENTRADAS_REALIZADAS',

    NR_PONT_OBR_FIM_ENTRADAS_REALIZADAS INT
        COMMENT 'Pontuação final do tema 5, Uso Consciente do Crédito, no cenário ENTRADAS_REALIZADAS',

    FL_PONTUACAO_COMPLETA_ENTRADAS_REALIZADAS STRING
        COMMENT 'Indica se as cinco pontuações foram calculadas no cenário ENTRADAS_REALIZADAS: S=completa; N=incompleta',

    NR_PONT_MAX_ENTRADAS_REALIZADAS INT
        COMMENT 'Maior pontuação final obtida no cenário ENTRADAS_REALIZADAS',

    QT_TEMAS_PONT_MAX_ENTRADAS_REALIZADAS INT
        COMMENT 'Quantidade de temas com a maior pontuação no cenário ENTRADAS_REALIZADAS',

    CD_TEMA_VENCEDOR_ENTRADAS_REALIZADAS INT
        COMMENT 'Código do tema vencedor no cenário ENTRADAS_REALIZADAS: 1=Categorização dos Gastos; 2=Gestão de Orçamento; 3=Consumo Planejado; 4=Formação de Reserva; 5=Uso Consciente do Crédito; 9=Empate; NULL=pontuação incompleta',

    TX_TEMA_VENCEDOR_ENTRADAS_REALIZADAS STRING
        COMMENT 'Descrição correspondente ao tema vencedor no cenário ENTRADAS_REALIZADAS'

)
COMMENT 'Resultado negocial do Radar Financeiro por cliente, contendo resultado oficial e os dois cenários calculados atualmente'
STORED AS PARQUET;
""")

# Inserir a fotografia completa na tabela recém-criada.
spark.sql("""
INSERT INTO TABLE ANA_RADAR_FIN_CLI
SELECT * FROM rts_resultado
""")
print(f'Fotografia inserida: {metricas_resultado["QT_RESULTADO"]} clientes.')


In [ ]:
%%time
%%spark
# Confirmar a quantidade, a chave e o contrato da fotografia publicada.
assert spark.table('ANA_RADAR_FIN_CLI').dtypes == contrato_esperado, 'Contrato divergente no readback.'
spark.sql(f"""
SELECT COUNT(*) AS QT_PUBLICADA,
       ASSERT_TRUE(COUNT(*) = {metricas_resultado["QT_RESULTADO"]}, 'Quantidade divergente no readback.'),
       ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT CD_CLI), 'Cliente nulo ou duplicado no readback.')
FROM ANA_RADAR_FIN_CLI
""").show()
spark.sql("UNCACHE TABLE rts_resultado")
spark.sql("UNCACHE TABLE rts_publico")
resultado_validado = False
